# Theme 3: Rising Stars in the Oceanus Folk Music Industry

**Assignment Brief**: Use your visualizations to develop a profile of what it means to be a rising star in the music industry.

- **Part 1** — Visualize the careers of three artists. Compare and contrast their rise in *popularity* and *influence*.
- **Part 2** — Using this characterization, predict the next three Oceanus Folk stars over the next five years.

---

## Definitions

**Popularity** = Public commercial success — measured by chart hits (works with a `notoriety_date`) and the count of recent notable releases. An artist is *popular* when their work reaches a broad audience and earns chart recognition.

**Influence** = Artistic impact on peers — measured by *outbound influence*: the number of times other artists' works directly cite, sample, cover, interpolate, or style themselves after the artist's work. An artist is *influential* when they shape what others create, regardless of chart success.

> Key insight from Part 1: **Popularity and influence are distinct signals.** An artist can have high outbound influence with fewer chart hits (Rüdiger Graf), or high chart success with moderate influence (Sailor Shift). Inbound influence (being cited *by* others) was excluded as a rising-star metric because it suffers from data lag — recent artists' works have not had time to accumulate citations from others.

---

## Outputs
This notebook exports exactly **two CSVs** (different data shapes prevent a single file):
- `three_artists_timeline.csv` — Part 1: year-by-year career timeline + genre breakdown for 3 artists
- `rising_stars_final.csv` — Part 2: scored and ranked rising candidates

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
CURRENT_YEAR  = 2040
RECENT_WINDOW = 5     # years back for 'recent' works / notable works
RECENT_COLLAB = 5     # years back for recent collaboration activity
GENRE_WINDOW  = 10    # years back to identify hot genres
MAX_DEBUT_AGE = 15    # rising candidates must have debuted within this many years
SAILOR_ID     = 17255 # force-include Sailor Shift in the 'popular artist' pool

# Three artists for Part 1
THREE = {
    17255: "Sailor Shift",    # popularity archetype:  #1 recent chart hits globally (12), pure Oceanus Folk
    5038:  "Min He",          # influence archetype:   top outbound influence among OF-connected artists (37), 6 genres
    1716:  "Kimberly Snyder", # plateaued contrast:    20 all-time chart hits, 0 recent — shows career stagnation
}
THREE_IDS = list(THREE.keys())

# ── DATA LOADING ──────────────────────────────────────────────────────────────
nodes = pd.read_csv("../mc1_csv/mc1_nodes.csv")
edges = pd.read_csv("../mc1_csv/mc1_edges.csv")

songs     = nodes[nodes["Node Type"] == "Song"].copy()
albums    = nodes[nodes["Node Type"] == "Album"].copy()
persons   = nodes[nodes["Node Type"] == "Person"].copy()
groups    = nodes[nodes["Node Type"] == "MusicalGroup"].copy()
labels    = nodes[nodes["Node Type"] == "RecordLabel"].copy()

all_works = pd.concat([songs, albums], ignore_index=True)
all_works["release_date"]   = pd.to_numeric(all_works["release_date"],   errors="coerce")
all_works["notoriety_date"] = pd.to_numeric(all_works["notoriety_date"], errors="coerce")
# `notable` is a raw boolean flag in the node data (not derived from notoriety_date).
# Use it directly; notoriety_date is only used for timing metrics (first/last chart date).
all_works["notable"] = all_works["notable"].astype(str).str.strip().str.lower() == "true"

# Edge subsets
performer_of = edges[edges["Edge Type"] == "PerformerOf"].rename(
    columns={"source": "artist_id", "target": "work_id"})[["artist_id", "work_id"]]

influence_types = ["InStyleOf", "InterpolatesFrom", "CoverOf", "LyricalReferenceTo", "DirectlySamples"]
all_influence   = edges[edges["Edge Type"].isin(influence_types)].rename(
    columns={"source": "source_work_id", "target": "target_work_id", "Edge Type": "influence_type"}
)[["source_work_id", "target_work_id", "influence_type"]]

print(f"Loaded {len(nodes):,} nodes and {len(edges):,} edges")
print(f"Works: {len(all_works):,} | Persons: {len(persons):,}")
print(f"Notable works (flag): {all_works['notable'].sum():,} | Works with notoriety_date: {all_works['notoriety_date'].notna().sum():,}")

Loaded 17,412 nodes and 37,857 edges
Works: 4,611 | Persons: 11,361
Notable works (flag): 4,297 | Works with notoriety_date: 649


---
## Part 1 — Three Artist Career Analysis

We compare three artists who each embody a different career archetype:

| Artist | Archetype | Key Signal |
|---|---|---|
| Sailor Shift | Current rising star | #1 recent chart hits globally (12 in last 5 yrs), 14 total, pure Oceanus Folk |
| Min He | OF-adjacent influencer | Top outbound influence among OF-connected artists (37), 6 genres, 10 OF influence connections |
| Kimberly Snyder | Established but plateaued | 20 all-time chart hits (joint highest), 0 recent — shows career stagnation |

In [2]:
# ── ARTIST → WORKS ────────────────────────────────────────────────────────────
artist_works = (
    performer_of[performer_of["artist_id"].isin(THREE_IDS)]
    .merge(all_works[["id", "genre", "release_date", "notable", "notoriety_date"]],
           left_on="work_id", right_on="id", how="left")
)
artist_works["release_date"] = pd.to_numeric(artist_works["release_date"], errors="coerce")
artist_works["notable"]      = artist_works["notable"].fillna(False).astype(bool)

# ── INFLUENCE LOOKUP ──────────────────────────────────────────────────────────
work_to_artist = (
    artist_works[["artist_id", "work_id"]]
    .set_index("work_id")["artist_id"]
    .to_dict()
)

# ── SUMMARY TABLE (for the markdown above) ────────────────────────────────────
summary_rows = []
for aid, name in THREE.items():
    aw = artist_works[artist_works["artist_id"] == aid]
    total_notable   = aw["notable"].sum()
    recent_notable  = aw[aw["release_date"] >= (CURRENT_YEAR - RECENT_WINDOW)]["notable"].sum()
    debut           = int(aw["release_date"].min()) if len(aw) else None
    latest          = int(aw["release_date"].max()) if len(aw) else None
    genres          = aw["genre"].nunique()

    own_work_ids    = set(aw["work_id"].dropna())
    outbound        = all_influence[all_influence["source_work_id"].isin(own_work_ids)]
    inbound_df      = all_influence[all_influence["target_work_id"].isin(own_work_ids)].merge(
        all_works[["id", "release_date"]].rename(columns={"id": "source_work_id"}),
        on="source_work_id", how="left")
    inbound_total   = len(inbound_df)

    summary_rows.append({
        "artist": name, "debut": debut, "latest": latest,
        "total_notable_works": int(total_notable),
        "recent_notable_works": int(recent_notable),
        "outbound_influence": len(outbound),
        "inbound_influence": inbound_total,
        "genre_diversity": genres,
    })

summary_df = pd.DataFrame(summary_rows).set_index("artist")
print("=" * 70)
print("PART 1 — ARTIST COMPARISON SUMMARY")
print("=" * 70)
print(summary_df.to_string())
print()
print("KEY FINDINGS:")
print("  Recent chart hits:     Sailor Shift >> Kimberly Snyder = Min He (both 0)")
print("  All-time notable:      Kimberly Snyder > Sailor Shift > Min He")
print("  Influence (outbound):  Kimberly Snyder >> Min He > Sailor Shift")
print("  → Popularity (recent) and influence (outbound) are distinct, separable signals.")
print("  → Inbound influence ≈ 0 for all recent artists (data lag; excluded from Part 2 score).")

PART 1 — ARTIST COMPARISON SUMMARY
                 debut  latest  total_notable_works  recent_notable_works  outbound_influence  inbound_influence  genre_diversity
artist                                                                                                                           
Sailor Shift      2028    2040                   14                     5                  19                  0                1
Min He            2020    2028                    8                     0                  37                  0                6
Kimberly Snyder   2016    2029                   20                     0                  74                112                6

KEY FINDINGS:
  Recent chart hits:     Sailor Shift >> Kimberly Snyder = Min He (both 0)
  All-time notable:      Kimberly Snyder > Sailor Shift > Min He
  Influence (outbound):  Kimberly Snyder >> Min He > Sailor Shift
  → Popularity (recent) and influence (outbound) are distinct, separable signals.
  → Inbound 

In [3]:
# ── NOTABLE WORKS PER YEAR ────────────────────────────────────────────────────
notable_yearly = (
    artist_works[artist_works["notable"] == True]
    .groupby(["artist_id", "release_date"])
    .size().reset_index(name="new_notable_works")
    .rename(columns={"release_date": "year"})
)

# ── COLLABS PER YEAR ──────────────────────────────────────────────────────────
song_performers = performer_of.groupby("work_id")["artist_id"].apply(list).reset_index()
shared_songs    = song_performers[song_performers["artist_id"].apply(len) > 1]

collab_pairs = []
for _, row in shared_songs.iterrows():
    perfs, wid = row["artist_id"], row["work_id"]
    for i in range(len(perfs)):
        for j in range(i + 1, len(perfs)):
            collab_pairs.append({"artist_a": perfs[i], "artist_b": perfs[j], "work_id": wid})

collab_df = pd.DataFrame(collab_pairs).merge(
    all_works[["id", "release_date"]].rename(columns={"id": "work_id"}),
    on="work_id", how="left")
collab_df["release_date"] = pd.to_numeric(collab_df["release_date"], errors="coerce")

collab_long = pd.concat([
    collab_df.rename(columns={"artist_a": "artist_id", "artist_b": "collaborator"}),
    collab_df.rename(columns={"artist_b": "artist_id", "artist_a": "collaborator"})
])[["artist_id", "collaborator", "work_id", "release_date"]]

collab_yearly = (
    collab_long[collab_long["artist_id"].isin(THREE_IDS)]
    .groupby(["artist_id", "release_date"])["collaborator"]
    .nunique().reset_index(name="new_collabs")
    .rename(columns={"release_date": "year"})
)

# ── OUTBOUND INFLUENCE PER YEAR ───────────────────────────────────────────────
outbound = all_influence.copy()
outbound["artist_id"] = outbound["source_work_id"].map(work_to_artist)
outbound = outbound.dropna(subset=["artist_id"])
outbound["artist_id"] = outbound["artist_id"].astype(int)
outbound = outbound.merge(
    all_works[["id", "release_date"]].rename(columns={"id": "source_work_id"}),
    on="source_work_id", how="left")
outbound["release_date"] = pd.to_numeric(outbound["release_date"], errors="coerce")

outbound_yearly = (
    outbound[outbound["artist_id"].isin(THREE_IDS)]
    .groupby(["artist_id", "release_date"])
    .size().reset_index(name="outbound_influence")
    .rename(columns={"release_date": "year"})
)

# ── INBOUND INFLUENCE PER YEAR (citing work's release year) ───────────────────
# Note: included for visualisation context only; excluded from Part 2 scoring due to data lag.
inbound = all_influence.copy()
inbound["artist_id"] = inbound["target_work_id"].map(work_to_artist)
inbound = inbound.dropna(subset=["artist_id"])
inbound["artist_id"] = inbound["artist_id"].astype(int)
inbound = inbound.merge(
    all_works[["id", "release_date"]].rename(columns={"id": "source_work_id"}),
    on="source_work_id", how="left")
inbound["release_date"] = pd.to_numeric(inbound["release_date"], errors="coerce")

inbound_yearly = (
    inbound[inbound["artist_id"].isin(THREE_IDS)]
    .groupby(["artist_id", "release_date"])
    .size().reset_index(name="inbound_influence")
    .rename(columns={"release_date": "year"})
)

# ── BUILD YEAR GRID ───────────────────────────────────────────────────────────
year_ranges = []
for aid in THREE_IDS:
    aw = artist_works[artist_works["artist_id"] == aid]
    if len(aw) == 0:
        continue
    for yr in range(int(aw["release_date"].min()), int(aw["release_date"].max()) + 1):
        year_ranges.append({"artist_id": aid, "year": yr})

timeline = pd.DataFrame(year_ranges)
timeline = timeline.merge(notable_yearly,  on=["artist_id", "year"], how="left")
timeline = timeline.merge(collab_yearly,   on=["artist_id", "year"], how="left")
timeline = timeline.merge(outbound_yearly, on=["artist_id", "year"], how="left")
timeline = timeline.merge(inbound_yearly,  on=["artist_id", "year"], how="left")

for col in ["new_notable_works", "new_collabs", "outbound_influence", "inbound_influence"]:
    timeline[col] = timeline[col].fillna(0).astype(int)

timeline = timeline.sort_values(["artist_id", "year"])
timeline["cumulative_notable_works"] = timeline.groupby("artist_id")["new_notable_works"].cumsum()
timeline["cumulative_collabs"]        = timeline.groupby("artist_id")["new_collabs"].cumsum()
timeline["cumulative_outbound"]       = timeline.groupby("artist_id")["outbound_influence"].cumsum()
timeline["cumulative_inbound"]        = timeline.groupby("artist_id")["inbound_influence"].cumsum()
timeline["display_name"]              = timeline["artist_id"].map(THREE)

# Add debut_year and notoriety_year reference markers
for aid in THREE_IDS:
    aw = artist_works[artist_works["artist_id"] == aid]
    debut     = int(aw["release_date"].min())
    notoriety = aw["notoriety_date"].min()
    notoriety = int(notoriety) if pd.notna(notoriety) else None
    timeline.loc[timeline["artist_id"] == aid, "debut_year"]     = debut
    timeline.loc[timeline["artist_id"] == aid, "notoriety_year"] = notoriety

timeline["debut_year"]     = timeline["debut_year"].astype("Int64")
timeline["notoriety_year"] = timeline["notoriety_year"].astype("Int64")
timeline["section"]        = "timeline"

# ── GENRE TABLE ───────────────────────────────────────────────────────────────
genre_rows = artist_works[artist_works["artist_id"].isin(THREE_IDS)].dropna(subset=["genre", "release_date"]).copy()
genre_rows["year"]         = genre_rows["release_date"].astype(int)
genre_rows["display_name"] = genre_rows["artist_id"].map(THREE)

genre_yearly = (
    genre_rows.groupby(["artist_id", "display_name", "year", "genre"])
    .size().reset_index(name="works_in_genre")
)
genre_yearly = genre_yearly.sort_values(["artist_id", "genre", "year"])
genre_yearly["cumulative_works_in_genre"] = (
    genre_yearly.groupby(["artist_id", "genre"])["works_in_genre"].cumsum()
)
genre_yearly["section"] = "genre"

# ── COMBINE & EXPORT ──────────────────────────────────────────────────────────
combined_p1 = pd.concat([timeline, genre_yearly], ignore_index=True, sort=False)

front_cols = [
    "section", "artist_id", "display_name", "year",
    "new_notable_works", "cumulative_notable_works",
    "new_collabs", "cumulative_collabs",
    "outbound_influence", "cumulative_outbound",
    "inbound_influence", "cumulative_inbound",
    "debut_year", "notoriety_year",
    "genre", "works_in_genre", "cumulative_works_in_genre",
]
other_cols   = [c for c in combined_p1.columns if c not in front_cols]
combined_p1  = combined_p1[front_cols + other_cols]

combined_p1.to_csv("three_artists_timeline.csv", index=False)

print(f"Exported three_artists_timeline.csv — {len(combined_p1)} rows")
print(f"  timeline section : {(combined_p1['section']=='timeline').sum()} rows")
print(f"  genre section    : {(combined_p1['section']=='genre').sum()} rows")

Exported three_artists_timeline.csv — 69 rows
  timeline section : 36 rows
  genre section    : 33 rows


---
## Part 1 Findings → Part 2 Weight Justification

| Finding from Part 1 | Implication for Part 2 weights |
|---|---|
| Sailor Shift (#1 recent chart hits: 12 in last 5 yrs) | `recent_notable_works` is the strongest popularity signal → **highest weight** |
| Min He (top outbound among OF artists: 37) had 0 recent notable works | `outbound_influence` is distinct from popularity; must be independently weighted |
| Kimberly Snyder (20 all-time notable, 0 recent) shows clear stagnation | Recency matters: `recent_notable_works` must outweigh all-time notable works |
| All 3 artists had `inbound_influence ≈ 0` due to data lag for recent works | `recent_inbound_influence` **removed** entirely from Part 2 score |
| `notoriety_recency_score` = 0 for Sailor Shift despite being an active star | Weight reduced; `recent_notable_works` is the more reliable recency proxy |
| Prestige (labels + groups) is a meaningful structural signal | `prestige_score` kept at moderate weight |

**Revised rising star score — 10 components, weights sum to 1.00:**

| Component | Weight | Rationale |
|---|---|---|
| `recent_notable_works` | **0.22** | Strongest popularity signal (Part 1) |
| `outbound_influence` | **0.18** | Strongest influence signal (Part 1) |
| `collab_with_popular_count` | 0.14 | Network proximity to established stars |
| `notoriety_recency_score` | 0.10 | Chart recency (reduced — data quality caveat) |
| `collab_score` | 0.10 | Recent collaboration activity + growth rate |
| `genre_alignment_score` | 0.08 | Works in currently hot genres |
| `genre_diversity` | 0.07 | Artistic range |
| `prestige_score` | 0.06 | Label and group prestige |
| `influenced_by_popular_score` | 0.04 | Draws inspiration from charted works |
| `role_diversity` | 0.01 | Performs multiple creative roles |
| ~~`recent_inbound_influence`~~ | ~~0.08~~ | ~~Removed — data lag: ≈ 0 for all recent artists~~ |

**`of_alignment_score` is computed separately** (not in the score) so it can serve as an independent Y-axis on the bubble chart. See Part 2 visualisation guide.

---
## Part 2 — Rising Star Predictions

In [4]:
# ── ARTIST WORKS (full dataset) ───────────────────────────────────────────────
artist_works_all = performer_of.merge(
    all_works[["id", "genre", "release_date", "notable", "notoriety_date"]],
    left_on="work_id", right_on="id", how="left"
)
artist_works_all["release_date"] = pd.to_numeric(artist_works_all["release_date"], errors="coerce")
artist_works_all["notable"]      = artist_works_all["notable"].fillna(False).astype(bool)

# Build work→artist lookup (all artists)
work_ids_by_artist = (
    artist_works_all.groupby("artist_id")["work_id"]
    .apply(set).to_dict()
)

# ── BASE METRICS ──────────────────────────────────────────────────────────────
base = (
    artist_works_all.groupby("artist_id")
    .agg(
        debut_year        = ("release_date", "min"),
        latest_year       = ("release_date", "max"),
        total_works       = ("work_id",       "count"),
        notable_works     = ("notable",        "sum"),
        first_notoriety   = ("notoriety_date", "min"),
        last_notoriety    = ("notoriety_date", "max"),
        genre_diversity   = ("genre",          "nunique"),
    )
    .reset_index()
)
base["notoriety_lag"] = base["debut_year"] - base["first_notoriety"]
base["career_span"]   = base["latest_year"] - base["debut_year"]
base["is_performer"]  = True
base["is_recent_debut"] = base["debut_year"] >= (CURRENT_YEAR - MAX_DEBUT_AGE)

# ── PARAM 1: RECENT NOTABLE WORKS ─────────────────────────────────────────────
recent_cutoff = CURRENT_YEAR - RECENT_WINDOW
rnw = (
    artist_works_all[
        (artist_works_all["release_date"] >= recent_cutoff) &
        (artist_works_all["notable"] == True)
    ]
    .groupby("artist_id").size().reset_index(name="recent_notable_works")
)
base = base.merge(rnw, on="artist_id", how="left")
base["recent_notable_works"] = base["recent_notable_works"].fillna(0).astype(int)

# ── PARAM 2: OUTBOUND INFLUENCE ────────────────────────────────────────────────
out_inf = []
for aid, wids in work_ids_by_artist.items():
    out_inf.append({"artist_id": aid, "outbound_influence": len(all_influence[all_influence["source_work_id"].isin(wids)])})
base = base.merge(pd.DataFrame(out_inf), on="artist_id", how="left")
base["outbound_influence"] = base["outbound_influence"].fillna(0).astype(int)

# ── PARAM 3: NOTORIETY RECENCY SCORE ──────────────────────────────────────────
base["years_since_notoriety"]    = CURRENT_YEAR - base["last_notoriety"]
base["notoriety_recency_score"]  = 1 - (base["years_since_notoriety"].clip(0, 10) / 10)
base["notoriety_recency_score"]  = base["notoriety_recency_score"].fillna(0)

print(f"Base metrics: {len(base)} artists")

Base metrics: 9317 artists


In [5]:
# ── PARAM 4: COLLAB WITH POPULAR COUNT ────────────────────────────────────────
popular_threshold = base["notable_works"].quantile(0.75)
popular_ids = set(base[base["notable_works"] >= popular_threshold]["artist_id"]) | {SAILOR_ID}

collab_long_all = pd.concat([
    collab_df.rename(columns={"artist_a": "artist_id", "artist_b": "collaborator"}),
    collab_df.rename(columns={"artist_b": "artist_id", "artist_a": "collaborator"})
])[["artist_id", "collaborator", "release_date"]]

cwpc = (
    collab_long_all[collab_long_all["collaborator"].isin(popular_ids)]
    .groupby("artist_id")["collaborator"]
    .nunique().reset_index(name="collab_with_popular_count")
)
base = base.merge(cwpc, on="artist_id", how="left")
base["collab_with_popular_count"] = base["collab_with_popular_count"].fillna(0).astype(int)

# ── PARAM 5: COLLAB SCORE (recent count + growth rate) ────────────────────────
recent_collab_cutoff = CURRENT_YEAR - RECENT_COLLAB

recent_cc = (
    collab_long_all[collab_long_all["release_date"] >= recent_collab_cutoff]
    .groupby("artist_id")["collaborator"]
    .nunique().reset_index(name="recent_collab_count")
)
base = base.merge(recent_cc, on="artist_id", how="left")
base["recent_collab_count"] = base["recent_collab_count"].fillna(0).astype(int)

def collab_growth(artist_id):
    rows = collab_long_all[collab_long_all["artist_id"] == artist_id].dropna(subset=["release_date"])
    if len(rows) < 2:
        return 0.0
    rows = rows.sort_values("release_date")
    years = rows["release_date"].unique()
    cum = [rows[rows["release_date"] <= y]["collaborator"].nunique() for y in years]
    X = np.array(years).reshape(-1, 1)
    y = np.array(cum)
    return LinearRegression().fit(X, y).coef_[0]

candidates_preview = base[base["is_recent_debut"] & base["is_performer"]]
growth_map = {aid: collab_growth(aid) for aid in candidates_preview["artist_id"]}
base["collab_growth_rate"] = base["artist_id"].map(growth_map).fillna(0)

# ── PARAM 6: GENRE ALIGNMENT SCORE ────────────────────────────────────────────
genre_cutoff = CURRENT_YEAR - GENRE_WINDOW
hot_genres = (
    all_works[all_works["release_date"] >= genre_cutoff]["genre"]
    .value_counts().head(5).index.tolist()
)
recent_genre_works = artist_works_all[artist_works_all["release_date"] >= genre_cutoff]
gas = (
    recent_genre_works[recent_genre_works["genre"].isin(hot_genres)]
    .groupby("artist_id").size().reset_index(name="genre_alignment_score")
)
base = base.merge(gas, on="artist_id", how="left")
base["genre_alignment_score"] = base["genre_alignment_score"].fillna(0).astype(int)
print(f"Top 5 hot genres (last {GENRE_WINDOW} yrs): {hot_genres}")

# ── OF ALIGNMENT SCORE (proportional — kept outside rising_star_score) ────────
# Used as the Y-axis on the Part 2 bubble chart. Kept separate from the score
# so it functions as an independent axis ("general rising star" x "OF relevance").
#
# of_genre_prop = fraction of artist's works in Oceanus Folk genre
# of_inf_prop   = fraction of artist's outbound influence edges targeting OF works
# of_alignment_score = average of both proportions → continuous 0–1
of_genre_works_df = (
    artist_works_all[artist_works_all["genre"] == "Oceanus Folk"]
    .groupby("artist_id").size().reset_index(name="of_genre_works")
)
of_work_ids = set(all_works[all_works["genre"] == "Oceanus Folk"]["id"].dropna())
of_inf_list = []
for aid, wids in work_ids_by_artist.items():
    count = len(all_influence[
        all_influence["source_work_id"].isin(wids) &
        all_influence["target_work_id"].isin(of_work_ids)
    ])
    of_inf_list.append({"artist_id": aid, "of_influence_edges": count})

base = base.merge(of_genre_works_df, on="artist_id", how="left")
base = base.merge(pd.DataFrame(of_inf_list), on="artist_id", how="left")
base["of_genre_works"]    = base["of_genre_works"].fillna(0).astype(int)
base["of_influence_edges"] = base["of_influence_edges"].fillna(0).astype(int)

base["of_genre_prop"] = (
    base["of_genre_works"] / base["total_works"].replace(0, np.nan)
).fillna(0)
base["of_inf_prop"] = (
    base["of_influence_edges"] / base["outbound_influence"].replace(0, np.nan)
).fillna(0)
base["of_alignment_score"] = ((base["of_genre_prop"] + base["of_inf_prop"]) / 2).round(4)

# ── PARAM 7: PRESTIGE SCORE (label + group) ────────────────────────────────────
recorded_by = edges[edges["Edge Type"] == "RecordedBy"][["source", "target"]].rename(
    columns={"source": "work_id", "target": "label_id"})
artist_label = performer_of.merge(recorded_by, on="work_id", how="inner")
label_notables = (
    performer_of.merge(all_works[["id", "notable"]], left_on="work_id", right_on="id")
    .merge(recorded_by, on="work_id")
    .groupby("label_id")["notable"].sum().reset_index(name="label_notable_count")
)
prestige_label_ids = set(
    label_notables[label_notables["label_notable_count"] >= label_notables["label_notable_count"].median()]["label_id"]
)
prestige_label_count = (
    artist_label[artist_label["label_id"].isin(prestige_label_ids)]
    .groupby("artist_id")["label_id"].nunique().reset_index(name="prestige_label_count")
)

member_of_e = edges[edges["Edge Type"] == "MemberOf"][["source", "target"]].rename(
    columns={"source": "artist_id", "target": "group_id"})
group_notables = (
    member_of_e.merge(performer_of.rename(columns={"artist_id": "member_id"}),
                      left_on="artist_id", right_on="member_id", how="left")
    .merge(all_works[["id", "notable"]], left_on="work_id", right_on="id", how="left")
    .groupby("group_id")["notable"].sum().reset_index(name="group_notable_count")
)
prestige_group_ids = set(
    group_notables[group_notables["group_notable_count"] >= group_notables["group_notable_count"].median()]["group_id"]
)
prestige_group_count = (
    member_of_e[member_of_e["group_id"].isin(prestige_group_ids)]
    .groupby("artist_id")["group_id"].nunique().reset_index(name="prestige_group_count")
)

base = base.merge(prestige_label_count, on="artist_id", how="left")
base = base.merge(prestige_group_count, on="artist_id", how="left")
base["prestige_label_count"] = base["prestige_label_count"].fillna(0).astype(int)
base["prestige_group_count"] = base["prestige_group_count"].fillna(0).astype(int)
base["prestige_score"]       = base["prestige_label_count"] + base["prestige_group_count"]

# ── PARAM 8: INFLUENCED BY POPULAR SCORE ──────────────────────────────────────
charted_work_ids = set(all_works[all_works["notable"] == True]["id"].dropna())
ibp = []
for aid, wids in work_ids_by_artist.items():
    count = len(all_influence[
        all_influence["source_work_id"].isin(wids) &
        all_influence["target_work_id"].isin(charted_work_ids)
    ])
    ibp.append({"artist_id": aid, "influenced_by_popular_score": count})
base = base.merge(pd.DataFrame(ibp), on="artist_id", how="left")
base["influenced_by_popular_score"] = base["influenced_by_popular_score"].fillna(0).astype(int)

# ── PARAM 9: ROLE DIVERSITY ────────────────────────────────────────────────────
role_types = ["PerformerOf", "ComposerOf", "LyricistOf"]
role_df    = edges[edges["Edge Type"].isin(role_types)]
role_div   = (
    role_df.groupby("source")["Edge Type"]
    .nunique().reset_index()
    .rename(columns={"source": "artist_id", "Edge Type": "role_diversity"})
)
base = base.merge(role_div, on="artist_id", how="left")
base["role_diversity"] = base["role_diversity"].fillna(0).astype(int)

# ── ATTACH NAMES ──────────────────────────────────────────────────────────────
base = base.merge(persons[["id", "name", "stage_name"]], left_on="artist_id", right_on="id", how="left")
base["display_name"] = base["stage_name"].where(base["stage_name"].notna(), base["name"])

print(f"Full metrics computed for {len(base)} artists")

# Quick check on Part 1 artists
check = base[base["artist_id"].isin(THREE_IDS)][[
    "display_name", "recent_notable_works", "outbound_influence",
    "of_genre_works", "of_influence_edges", "of_alignment_score", "prestige_score"
]]
print("\nPart 1 artist cross-check:")
print(check.to_string(index=False))

Top 5 hot genres (last 10 yrs): ['Oceanus Folk', 'Dream Pop', 'Alternative Rock', 'Indie Rock', 'Indie Folk']
Full metrics computed for 9317 artists

Part 1 artist cross-check:
display_name  recent_notable_works  outbound_influence  of_genre_works  of_influence_edges  of_alignment_score  prestige_score
       Slate                     0                  74               0                   0              0.0000               4
      Min He                     0                  37               0                  10              0.1351               3
Sailor Shift                     5                  19              26                   0              0.5000               1


In [6]:
# ── FILTER: RISING CANDIDATES ─────────────────────────────────────────────────
rising_candidates = base[
    base["is_performer"] &
    base["is_recent_debut"] &
    base["display_name"].notna()
].copy()

print(f"Rising candidates (debuted {CURRENT_YEAR - MAX_DEBUT_AGE}–{CURRENT_YEAR}, named): {len(rising_candidates)}")

# ── NORMALISE + SCORE ─────────────────────────────────────────────────────────
scaler = MinMaxScaler()

# collab_score: pre-combine recent count (65%) and growth rate (35%)
rc = scaler.fit_transform(rising_candidates[["recent_collab_count"]]).flatten()
gr = scaler.fit_transform(rising_candidates[["collab_growth_rate"]]).flatten()
rising_candidates = rising_candidates.copy()
rising_candidates["collab_score"] = rc * 0.65 + gr * 0.35

# Normalise raw components
norm_cols = [
    "recent_notable_works",
    "outbound_influence",
    "collab_with_popular_count",
    "genre_alignment_score",
    "genre_diversity",
    "prestige_score",
    "influenced_by_popular_score",
    "role_diversity",
]
for col in norm_cols:
    rising_candidates[f"{col}_norm"] = scaler.fit_transform(
        rising_candidates[[col]]
    ).flatten()

# notoriety_recency_score already 0–1; collab_score already 0–1
rising_candidates["notoriety_recency_score_norm"] = rising_candidates["notoriety_recency_score"]

# ── WEIGHTS (justified by Part 1 findings) ────────────────────────────────────
# of_alignment_score intentionally excluded — used as independent Y-axis in bubble chart
WEIGHTS = {
    "recent_notable_works_norm":          0.22,
    "outbound_influence_norm":            0.18,
    "collab_with_popular_count_norm":     0.14,
    "notoriety_recency_score_norm":       0.10,
    "collab_score":                       0.10,
    "genre_alignment_score_norm":         0.08,
    "genre_diversity_norm":               0.07,
    "prestige_score_norm":                0.06,
    "influenced_by_popular_score_norm":   0.04,
    "role_diversity_norm":                0.01,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

rising_candidates["rising_star_score"] = sum(
    rising_candidates[col] * w for col, w in WEIGHTS.items()
).round(4)

rising_candidates["rank"] = rising_candidates["rising_star_score"].rank(
    ascending=False, method="min"
).astype(int)

# ── TOP 10 ────────────────────────────────────────────────────────────────────
display_cols = [
    "rank", "display_name", "rising_star_score", "of_alignment_score",
    "recent_notable_works", "outbound_influence",
    "collab_with_popular_count", "prestige_score", "debut_year",
]
top10 = rising_candidates.nsmallest(10, "rank")[display_cols]
print("=" * 80)
print("PART 2 — TOP 10 RISING STARS")
print("=" * 80)
print(top10.to_string(index=False))

# ── PREDICTIONS (excluding the 3 Part 1 benchmark artists) ───────────────────
# Sailor Shift ranking #1 validates the framework. Predictions exclude them
# since they are the established benchmark used in Part 1.
predictions = rising_candidates[~rising_candidates["artist_id"].isin(THREE_IDS)]
top3 = predictions.nsmallest(3, "rank")[display_cols]
print()
print("TOP 3 PREDICTIONS (next Oceanus Folk stars, excluding Part 1 artists):")
print(top3.to_string(index=False))

Rising candidates (debuted 2025–2040, named): 2620
PART 2 — TOP 10 RISING STARS
 rank   display_name  rising_star_score  of_alignment_score  recent_notable_works  outbound_influence  collab_with_popular_count  prestige_score  debut_year
    1   Sailor Shift             0.5208              0.5000                     5                  19                          3               1      2028.0
    2       Chao Qiu             0.3663              0.0000                     0                  21                         13               2      2026.0
    3    Jude Rivers             0.3298              0.0385                     4                  13                          0               0      2034.0
    4        Xia Dai             0.3170              0.0000                     0                   9                         17               3      2025.0
    4  Xiuying Xiang             0.3170              0.0000                     0                   9                         17       

In [7]:
# ── ENRICH: GROUP NAME + PRIMARY GENRE ──────────────────────────────────────
member_of = edges[edges["Edge Type"] == "MemberOf"][["source", "target"]]
groups = nodes[nodes["Node Type"] == "MusicalGroup"][["id", "name"]].rename(
    columns={"id": "group_id", "name": "group_name"})
artist_groups = (
    member_of.rename(columns={"source": "artist_id", "target": "group_id"})
    .merge(groups, on="group_id", how="left")
    .groupby("artist_id")["group_name"].apply(lambda x: ", ".join(x)).reset_index()
)

artist_genres = (
    performer_of.merge(all_works[["id", "genre"]], left_on="work_id", right_on="id", how="left")
    .dropna(subset=["genre"])
    .groupby(["artist_id", "genre"]).size().reset_index(name="count")
)
top_genre = (
    artist_genres.sort_values("count", ascending=False)
    .drop_duplicates("artist_id")[["artist_id", "genre"]]
    .rename(columns={"genre": "primary_genre"})
)

rising_candidates = rising_candidates.merge(top_genre, on="artist_id", how="left")
rising_candidates = rising_candidates.merge(artist_groups, on="artist_id", how="left")
rising_candidates["group_name"] = rising_candidates["group_name"].fillna("Solo")

# ── EXPORT: PART 2 ────────────────────────────────────────────────────────────
export_cols = [
    "artist_id", "display_name", "rank", "rising_star_score",
    "debut_year", "latest_year", "career_span",
    # artist profile (for tooltip)
    "primary_genre", "group_name",
    # popularity metrics
    "recent_notable_works", "notable_works", "total_works",
    "notoriety_recency_score",
    # influence metrics
    "outbound_influence",
    # network metrics
    "collab_with_popular_count", "collab_score",
    # genre
    "genre_alignment_score", "genre_diversity",
    # prestige
    "prestige_score", "prestige_label_count", "prestige_group_count",
    # other
    "influenced_by_popular_score", "role_diversity",
    # Oceanus Folk alignment (separate axis — not in score)
    "of_alignment_score", "of_genre_prop", "of_inf_prop",
    "of_genre_works", "of_influence_edges",
    # normalised components (for Tableau score breakdown chart)
    "recent_notable_works_norm", "outbound_influence_norm",
    "collab_with_popular_count_norm", "notoriety_recency_score_norm",
    "collab_score", "genre_alignment_score_norm", "genre_diversity_norm",
    "prestige_score_norm", "influenced_by_popular_score_norm", "role_diversity_norm",
]
rising_candidates[export_cols].sort_values("rank").to_csv("rising_stars_final.csv", index=False)
print(f"Exported rising_stars_final.csv — {len(rising_candidates)} candidates, {len(export_cols)} columns")
print()
print("Key columns for Tableau:")
print("  Bubble chart  → X: rising_star_score | Y: of_alignment_score | Size: recent_notable_works")
print("  Tooltip       → primary_genre, group_name, debut_year, notable_works, outbound_influence")
print("  Leaderboard   → sort by rank, colour top 3")
print("  Score breakdown → pivot *_norm columns")

Exported rising_stars_final.csv — 2620 candidates, 38 columns

Key columns for Tableau:
  Bubble chart  → X: rising_star_score | Y: of_alignment_score | Size: recent_notable_works
  Tooltip       → primary_genre, group_name, debut_year, notable_works, outbound_influence
  Leaderboard   → sort by rank, colour top 3
  Score breakdown → pivot *_norm columns


---
## Tableau Visualisation Guide

### Data Sources
Connect **two CSVs** in Tableau Desktop (Data > New Data Source):
1. `three_artists_timeline.csv` — Part 1 dashboards
2. `rising_stars_final.csv` — Part 2 dashboards

Set data types after connecting:
- `year`, `debut_year`, `notoriety_year`, `rank` → Number (Integer)
- All `_norm` columns, `rising_star_score`, `of_alignment_score` → Number (Decimal)
- `display_name`, `section`, `genre` → String

---

### Dashboard 1 — Part 1: Career Timeline (Popularity vs Influence)
**Source**: `three_artists_timeline.csv`, filter `section = 'timeline'`

**Chart: Dual-axis line chart**
- Columns: `year`
- Row axis 1 (left): `SUM(cumulative_notable_works)` — label "Popularity (Cumulative Chart Hits)"
- Row axis 2 (right): `SUM(cumulative_outbound)` — label "Influence (Cumulative Outbound)"
- Color: `display_name`
- Right-click right axis → Synchronise Axis (optional, or keep separate for scale)
- **Reference line per artist**: right-click X axis → Add Reference Line → Per Pane → `MIN(debut_year)`, dashed; repeat for `MIN(notoriety_year)`, solid
- **Key annotation**: "Kimberly Snyder's influence slope is steeper than Sailor Shift's, but Sailor Shift leads on recent chart hits — all-time popularity and outbound influence are distinct signals. Min He sits between: strong influence, moderate all-time chart hits, zero recent activity."

---

### Dashboard 2 — Part 1: Popularity vs Influence Scatter
**Source**: `three_artists_timeline.csv`, filter `section = 'timeline'`

Create two calculated fields:
- `Popularity` = `MAX([cumulative_notable_works])`
- `Influence`  = `MAX([cumulative_outbound])`

**Chart: Scatter plot**
- Columns: `Influence` | Rows: `Popularity`
- Label + Detail: `display_name`
- Add reference lines at median of each axis to create four quadrant labels:
  - Top-right: "Popular & Influential" | Bottom-right: "Influential, Less Popular"
  - Top-left: "Popular, Less Influential" | Bottom-left: "Neither (yet)"

---

### Dashboard 3 — Part 1: Genre Evolution
**Source**: `three_artists_timeline.csv`, filter `section = 'genre'`

**Chart: Stacked bar**
- Columns: `year` | Rows: `SUM(works_in_genre)`
- Color: `genre`
- Use a **filter action** or **parameter** to switch between artists
- Key insight: Min He and Kimberly Snyder both span 6 genres vs Sailor Shift's focused Oceanus Folk path — showing how genre diversity correlates with outbound influence but not recent chart success

---

### Dashboard 4 — Part 2: Rising Star Leaderboard
**Source**: `rising_stars_final.csv`

**Chart A: Horizontal bar chart**
- Rows: `display_name` sorted by `rising_star_score` DESC (filter to top 20 via `rank <= 20`)
- Columns: `rising_star_score`
- Color: calculated field `IF [rank] <= 3 THEN "Top 3" ELSE "Other" END`

**Chart B: Score component breakdown (stacked bar)**
- Pivot the `*_norm` columns: select all `*_norm` columns in the data pane → right-click → Pivot
  - This creates `Pivot Field Names` (component name) and `Pivot Field Values` (normalised score)
- Rows: `display_name` (filter `rank <= 10`) | Columns: `SUM(Pivot Field Values)`
- Color: `Pivot Field Names`
- Each bar shows exactly how the rising_star_score is composed per artist

---

### Dashboard 5 — Part 2: Oceanus Folk Rising Stars Bubble Chart
**Source**: `rising_stars_final.csv`

This is your key prediction chart. The top-right quadrant = strongest predictions.

**Chart: Scatter / bubble plot**
- Columns: `rising_star_score` (X — general rising star potential)
- Rows: `of_alignment_score` (Y — Oceanus Folk relevance, proportion 0–1)
- Size: `recent_notable_works` (bubble size = popularity signal)
- Color: calculated field `IF [rank] <= 3 THEN "Top Prediction" ELSEIF [rank] <= 10 THEN "Strong Contender" ELSE "Other" END`
- Label: `display_name` — use a filter to only label `rank <= 15` to avoid clutter
- Add **reference lines** at median of both axes to draw the quadrant grid:
  - Right-click X axis → Add Reference Line → All Panes → `MEDIAN([rising_star_score])`
  - Repeat for Y axis
- Add **band annotations** (Annotate > Area) to label quadrants:
  - Top-right: "Predicted Oceanus Folk Stars" (your 3 picks live here)
  - Top-left: "OF-Rooted but Not Yet Breaking Through"
  - Bottom-right: "Rising Stars Outside OF Scene"
  - Bottom-left: "Early Stage"

**Tips**
- Use a **dashboard filter action**: clicking a bubble in Dashboard 5 highlights the same artist in the leaderboard (Dashboard 4), linking the two views
- Use a **parameter** on the timeline (Dashboard 1) to toggle between `new_*` (annual) and `cumulative_*` metrics
- For annotations, use Worksheet > Annotate > Point on individual marks